# Swiss laying and breeding chickens: national trends and cantonal distribution

**Research notebook | Switzerland | 1975–2025**  
**Main outputs:** (1) national population time series; (2) 2025 choropleth by canton.

This notebook examines the reported number of **laying and breeding chickens**, using two extracts from the Swiss agricultural statistical service supplied for this analysis. The series is presented as a count of birds, not farms, egg production, poultry meat, or the broader chicken population.

The analysis is deliberately descriptive. It documents basic data-quality checks, compares canton totals to the national aggregate, and produces two export-ready figures. The canton map shows **absolute numbers**, not intensity relative to area, population, or agricultural holdings.

**Inputs** (in the same folder as this notebook):
- `Swiss_laying_and_breeding_chickens_by_year(1).csv`
- `Swiss_laying_and_breeding_chickens_by_canton_year(1).csv`

**Geographic boundaries:** 26 Swiss canton polygons, 2025 reference geometry, originating from Swiss Federal Statistical Office (FSO) generalized boundaries and distributed via [Swiss Geography Look-Ups](https://github.com/samkhavandi/Swiss_Geography_Look_Ups). A local GeoJSON copy is cached on first use. Internet access is needed only to download boundaries if they have not been cached. See the map section for details.

**Important:** Verify the original dataset title, extraction date, counting convention and methodological notes on the statistical service before citing these figures in a publication.

## 1. Set up the analytical environment

The workflow uses pandas for data manipulation, Matplotlib for the graphics, and GeoPandas for the cantonal boundaries. The colours are kept consistent between the two graphics, and all exported images are saved at 300 dpi.

In [ ]:
# Run the following once in the active Jupyter environment if necessary:
# %pip install pandas matplotlib geopandas requests

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import geopandas as gpd
import requests

ROOT = Path.cwd()  # Open/run this notebook from its containing folder.
OUTPUT = ROOT / "outputs"
OUTPUT.mkdir(exist_ok=True)

NATIONAL_CSV = ROOT / "data" / "swiss_laying_breeding_chickens_national.csv"
CANTON_CSV = ROOT / "data" / "swiss_laying_breeding_chickens_by_canton_year.csv"

# Graphic palette: restrained, publication-oriented red tones.
PRIMARY = "#A12F26"
DARK = "#641E19"
TEXT = "#303030"
GRID = "#E5E5E5"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelcolor": TEXT,
    "text.color": TEXT,
    "axes.edgecolor": GRID,
    "xtick.color": TEXT,
    "ytick.color": TEXT,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

## 2. Import and inspect the source files

The national file contains a percentage-change field. We calculate changes again when needed rather than assume that missing values are zero. The year variable is numeric and is used for chronological sorting. The earlier observations are spaced five years apart; **the time series should not be interpreted as having annual observations before 1996**.

In [ ]:
national = pd.read_csv(NATIONAL_CSV)
cantons = pd.read_csv(CANTON_CSV)

for df in (national, cantons):
    df["year"] = pd.to_numeric(df["year"], errors="raise").astype(int)
    df["laying_and_breeding_chickens"] = pd.to_numeric(
        df["laying_and_breeding_chickens"], errors="raise"
    ).astype(int)

national = national.sort_values("year").reset_index(drop=True)
cantons = cantons.sort_values(["year", "canton"]).reset_index(drop=True)

print(f"National observations: {len(national)}, {national.year.min()}–{national.year.max()}")
print(f"Canton-year observations: {len(cantons)} across {cantons.canton.nunique()} cantons")
display(national.head(), cantons.head())

## 3. Quality assurance and consistency checks

Each year must occur only once in the national dataset, and each canton-year combination must identify a single observation. Negative bird counts are not admissible. We also compare the sum across cantons with the reported national total in every year available in both files. Any inconsistency stops execution, preventing the creation of misleading graphics.

In [ ]:
assert not national["year"].duplicated().any(), "Duplicate years in national data."
assert not cantons.duplicated(["canton", "year"]).any(), "Duplicate canton-year records."
assert national["laying_and_breeding_chickens"].notna().all()
assert cantons["laying_and_breeding_chickens"].notna().all()
assert (national["laying_and_breeding_chickens"] >= 0).all()
assert (cantons["laying_and_breeding_chickens"] >= 0).all()
assert cantons["canton"].nunique() == 26, "Expected 26 Swiss cantons."

canton_sums = (cantons.groupby("year", as_index=False)
               ["laying_and_breeding_chickens"].sum()
               .rename(columns={"laying_and_breeding_chickens": "cantons_sum"}))
check = national[["year", "laying_and_breeding_chickens"]].merge(
    canton_sums, on="year", how="left", validate="one_to_one"
)
check["difference"] = check["cantons_sum"] - check["laying_and_breeding_chickens"]
assert check["cantons_sum"].notna().all(), "Canton totals missing for a national year."
assert (check["difference"] == 0).all(), "National and cantonal totals differ."
print("PASS: the cantonal sum matches the national figure for every reported year.")
display(check.tail())

## 4. National population of laying and breeding chickens

The figure plots the observed totals at their actual calendar years. Markers identify observed years, avoiding the impression that every intervening year has been measured. A vertical reference line identifies 2025, the year used for the spatial analysis. The visual representation is descriptive; it does not attempt to estimate trends statistically or attribute changes to policy, technology, or market developments.

In [ ]:
series = national.set_index("year")["laying_and_breeding_chickens"]
latest_year = 2025
latest_total = int(series.loc[latest_year])

fig, ax = plt.subplots(figsize=(11.6, 5.8))
ax.plot(series.index, series.values / 1_000_000,
        color=PRIMARY, linewidth=2.6, marker="o", markersize=3.5,
        markerfacecolor="white", markeredgewidth=1.15)
ax.scatter([latest_year], [latest_total / 1_000_000], color=DARK, s=52, zorder=5)
ax.annotate(f"2025: {latest_total:,.0f} birds",
            xy=(latest_year, latest_total / 1_000_000), xytext=(-10, 17),
            textcoords="offset points", ha="right", color=DARK, weight="bold")
ax.set(title="Laying and breeding chickens in Switzerland, 1975–2025",
       xlabel="Year", ylabel="Number of birds (millions)")
ax.set_xlim(1973, 2028)
ax.yaxis.set_major_locator(mticker.MaxNLocator(6))
ax.grid(axis="y", color=GRID, linewidth=0.9)
ax.set_axisbelow(True)
fig.text(0.10, 0.005,
         "Source: Swiss agricultural statistical service; supplied national series. Observed years only.",
         fontsize=9, color="#666666")
fig.tight_layout(rect=(0, 0.045, 1, 1))
fig.savefig(OUTPUT / "swiss_laying_breeding_chickens_trend.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
start_year = int(national.year.min())
start_total = int(series.loc[start_year])
percentage_change = (latest_total / start_total - 1) * 100
peak_year = int(series.idxmax())
peak_total = int(series.max())
print(f"{start_year}: {start_total:,} birds")
print(f"2025: {latest_total:,} birds")
print(f"Change ({start_year}–2025): {percentage_change:+.1f}%")
print(f"Highest observed count: {peak_total:,} birds ({peak_year})")
print("Note: 1975, 1980, 1985 and 1990 are spaced observations, not annual data.")

## 5. Canton distribution in 2025

We restrict the canton dataset to 2025 and calculate each canton's share of the Swiss total. The first table is a useful companion to the choropleth: unlike the map, it makes rankings and differences in absolute size explicit. The percentages use the national count of laying and breeding chickens as the denominator.

In [ ]:
map_year = 2025
c2025 = cantons.loc[cantons["year"].eq(map_year),
                    ["canton", "canton_name", "laying_and_breeding_chickens"]].copy()
assert len(c2025) == 26 and c2025["canton"].nunique() == 26
assert c2025["laying_and_breeding_chickens"].sum() == latest_total
c2025["share_pct"] = 100 * c2025["laying_and_breeding_chickens"] / latest_total
c2025 = c2025.sort_values("laying_and_breeding_chickens", ascending=False)
display(c2025.head(10).style.format({"laying_and_breeding_chickens": "{:,.0f}", "share_pct": "{:.1f}%"}))

## 6. Download canton boundaries and join to the data

The map uses the **2025-01-01** generalized canton boundaries (WGS 84 / EPSG:4326) distributed by the open-source [Swiss Geography Look-Ups project](https://github.com/samkhavandi/Swiss_Geography_Look_Ups), based on FSO geographic data. The boundaries are not derived from the poultry dataset.

The original input uses two-letter canton abbreviations (`ZH`, `BE`, etc.); the GeoJSON provides `canton_abbr`. We join these explicitly and verify that all 26 cantons match. The GeoJSON is stored locally after its first successful download; if your computer is offline, place a copy named `swiss_cantons_2025.geojson` beside the notebook.

For official boundary access, see [FSO GEOSTAT](https://statistik.admin.ch/en/daten/swiss-federal-statistics-geodata-geostat).

In [ ]:
GEOJSON_URL = (
    "https://raw.githubusercontent.com/samkhavandi/"
    "Swiss_Geography_Look_Ups/main/geometries/canton.geojson"
)
GEOJSON_PATH = ROOT / "swiss_cantons_2025.geojson"

if not GEOJSON_PATH.exists():
    try:
        response = requests.get(GEOJSON_URL, timeout=40)
        response.raise_for_status()
        # Confirm that the response is GeoJSON before writing it to disk.
        payload = response.json()
        assert payload.get("type") == "FeatureCollection"
        GEOJSON_PATH.write_text(response.text, encoding="utf-8")
        print(f"Downloaded canton boundaries to {GEOJSON_PATH.name}")
    except Exception as exc:
        raise RuntimeError(
            "The canton GeoJSON could not be retrieved. Connect to the internet, "
            "or download 'geometries/canton.geojson' from the repository "
            "and save it beside this notebook as 'swiss_cantons_2025.geojson'."
        ) from exc


# Load Swiss canton boundaries using GeoPandas.
geo = gpd.read_file(GEOJSON_PATH)

# Inspect the geographic dataset.
print("Available columns:", geo.columns.tolist())
print(geo[["canton_nr", "canton"]].head())

# Rename the canton identifier to match the statistical dataset.
geo = geo.rename(columns={"canton": "canton_abbr"})

# Standardize identifiers before joining the datasets.
geo["canton_abbr"] = (
    geo["canton_abbr"]
    .astype(str)
    .str.upper()
    .str.strip()
)

c2025["canton"] = (
    c2025["canton"]
    .astype(str)
    .str.upper()
    .str.strip()
)

# Verify geographic coverage and identifier uniqueness.
assert len(geo) == 26, "Expected 26 Swiss cantons."
assert geo["canton_abbr"].is_unique, "Duplicate canton identifiers."

# Merge 2025 poultry observations with canton boundaries.
mapped = geo.merge(
    c2025,
    left_on="canton_abbr",
    right_on="canton",
    how="left",
    validate="one_to_one",
    indicator=True
)

# Verify that every canton has a corresponding observation.
assert mapped["_merge"].eq("both").all(), (
    "Some cantons could not be matched: "
    + str(mapped.loc[mapped["_merge"] != "both", "canton_abbr"].tolist())
)

assert mapped["laying_and_breeding_chickens"].notna().all()

print("PASS: All 26 Swiss cantons matched successfully.")


## 7. Choropleth: laying and breeding chickens by canton, 2025

Darker shading indicates a larger **absolute number of birds**. The legend is expressed in thousands for readability, while the underlying analysis retains the original counts. A canton with a larger territory is not necessarily associated with more birds; map area and bird numbers measure different things. This figure should therefore be interpreted alongside the ranked table above.

In [ ]:
# Reproject to Swiss LV95 (metres) for an appropriate local map geometry.
plot_geo = mapped.to_crs(epsg=2056)
plot_geo["birds_thousands"] = plot_geo["laying_and_breeding_chickens"] / 1000

fig, ax = plt.subplots(figsize=(11.2, 7.0))
plot_geo.plot(column="birds_thousands", ax=ax, cmap="Reds", edgecolor="white",
              linewidth=0.7, legend=True,
              legend_kwds={"label": "Number of birds (thousands)", "shrink": 0.73})
ax.set_title("Laying and breeding chickens by Swiss canton, 2025", pad=15,
             fontsize=15, color=TEXT, weight="bold")
ax.set_axis_off()

# Two-letter abbreviations aid identification of smaller cantons.
for _, canton_row in plot_geo.iterrows():
    centre = canton_row.geometry.representative_point()
    ax.annotate(canton_row["canton_abbr"], xy=(centre.x, centre.y),
                ha="center", va="center", fontsize=7.2, color="#303030")

fig.text(0.07, 0.025,
         "Source: Swiss agricultural statistical service (2025); canton boundaries: FSO via Swiss Geography Look-Ups.",
         fontsize=8.5, color="#666666")
fig.tight_layout(rect=(0, 0.055, 1, 1))
fig.savefig(OUTPUT / "swiss_laying_breeding_chickens_canton_2025.png",
            dpi=300, bbox_inches="tight")
plt.show()

## 8. Export analytical table and reporting notes

The CSV export contains the canton code, canton name, 2025 count, and national share. The table and graphics can be cited together in a report or used as starting points for subsequent work on the geographic concentration of poultry production.

In [ ]:
export = c2025[["canton", "canton_name", "laying_and_breeding_chickens", "share_pct"]]
export.to_csv(OUTPUT / "swiss_laying_breeding_chickens_cantons_2025.csv", index=False)
print("Figures and canton table saved to:", OUTPUT.resolve())

## 9. Combine both graphics into a summary infographic

To conclude the analysis, this step combines the national time-series chart and the 2025 canton-level choropleth into a single infographic. This provides a compact visual summary of both the temporal evolution and the geographic distribution of Swiss laying and breeding chickens.
The infographic is intended as a presentation-ready output that can be used in reports, slides, or briefing materials.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.image as mpimg
from matplotlib.gridspec import GridSpec

# -------------------------------------------------------------------
# 9.1. Locate outputs relative to the repository (not a personal drive)
# -------------------------------------------------------------------
OUTPUT_DIR = OUTPUT
line_chart_path = OUTPUT_DIR / "swiss_laying_breeding_chickens_trend.png"
map_chart_path = OUTPUT_DIR / "swiss_laying_breeding_chickens_canton_2025.png"
infographic_path = OUTPUT_DIR / "swiss_laying_breeding_chickens_infographic.png"

for graphic in (line_chart_path, map_chart_path):
    if not graphic.is_file():
        raise FileNotFoundError(f"Missing graphic: {graphic}. Run Steps 4 and 7 first.")
print("PASS: Both graphics located successfully.")

# -------------------------------------------------------------------
# 9.2. Load the exported charts
# -------------------------------------------------------------------
line_img = mpimg.imread(line_chart_path)
map_img = mpimg.imread(map_chart_path)

# -------------------------------------------------------------------
# 9.3. Use the same source files as the analysis above
# -------------------------------------------------------------------
national = pd.read_csv(NATIONAL_CSV, encoding="utf-8-sig")
# The 2025 canton table is produced in Step 8; otherwise derive it
# from the original canton dataset to avoid dependency on saved extracts.
canton_path = OUTPUT_DIR / "swiss_laying_breeding_chickens_cantons_2025.csv"
if canton_path.exists():
    cantons = pd.read_csv(canton_path)
else:
    cantons = pd.read_csv(CANTON_CSV, encoding="utf-8-sig")
for df in (national, cantons):
    df.columns = df.columns.str.strip()
print("PASS: Source datasets loaded successfully.")

# -------------------------------------------------------------------
# 9.4. Calculate national indicators
# -------------------------------------------------------------------
value_col = "laying_and_breeding_chickens"
national["year"] = pd.to_numeric(national["year"], errors="raise")
national[value_col] = pd.to_numeric(national[value_col], errors="raise")
assert national["year"].is_unique, "Duplicate national years detected."
national = national.sort_values("year")
start_year = int(national["year"].min())
end_year = int(national["year"].max())
start_value = int(national.loc[national["year"].eq(start_year), value_col].iloc[0])
end_value = int(national.loc[national["year"].eq(end_year), value_col].iloc[0])
assert start_value > 0
pct_change = (end_value / start_value - 1) * 100
reference_year = 2025
reference_values = national.loc[national["year"].eq(reference_year), value_col]
assert len(reference_values) == 1, "Missing or repeated 2025 national observation."
national_2025 = int(reference_values.iloc[0])

# -------------------------------------------------------------------
# 9.5. Validate 2025 cantonal observations and prepare rankings
# -------------------------------------------------------------------
if "year" in cantons.columns:
    cantons["year"] = pd.to_numeric(cantons["year"], errors="raise")
    canton_2025 = cantons.loc[cantons["year"].eq(reference_year)].copy()
else:
    # No year column is expected in the Step 8 exported 2025-only table.
    canton_2025 = cantons.copy()

required = {"canton", "canton_name", value_col}
assert required.issubset(canton_2025.columns), "Missing canton fields."
canton_2025["canton"] = canton_2025["canton"].str.upper().str.strip()
canton_2025[value_col] = pd.to_numeric(canton_2025[value_col], errors="raise")
assert len(canton_2025) == 26, "Expected 26 canton records."
assert canton_2025["canton"].is_unique, "Duplicate canton identifiers."
assert canton_2025[value_col].notna().all(), "Missing poultry counts."
canton_sum = int(canton_2025[value_col].sum())
assert canton_sum == national_2025, (
    f"Cantonal sum {canton_sum:,} differs from national total {national_2025:,}."
)
top3 = canton_2025.sort_values(value_col, ascending=False).head(3).reset_index(drop=True)
top3_share = 100 * top3[value_col].sum() / national_2025
print(f"PASS: 2025 total {national_2025:,}; {start_year}-{end_year} change {pct_change:+.1f}%")
print("PASS: All 26 canton counts reconcile with the national total.")


In [ ]:
# -------------------------------------------------------------------
# 9.6. Define the infographic's visual identity
# -------------------------------------------------------------------

# Use a restrained colour palette suitable for research reports.
# The burgundy accents complement the two previously exported figures.

PRIMARY = "#A12F26"
BURGUNDY = "#641E19"
TEXT = "#303030"
MUTED = "#666666"
LIGHT = "#F5F5F5"
BORDER = "#E5E5E5"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "text.color": TEXT,
    "axes.titlecolor": TEXT,
    "figure.facecolor": "white",
    "savefig.facecolor": "white"
})


# -------------------------------------------------------------------
# 9.7. Construct the infographic layout
# -------------------------------------------------------------------

# Create a landscape-oriented infographic with sufficient space
# for the two figures, headline statistics, and source information.

fig = plt.figure(figsize=(18, 11), facecolor="white")

# Define the layout using a two-column grid.
# The middle row accommodates the previously exported graphics,
# while the bottom row contains the summary statistics.

gs = GridSpec(
    2, 2,
    figure=fig,
    height_ratios=[3.8, 1.25],
    width_ratios=[1, 1],
    hspace=0.15,
    wspace=0.06,
    left=0.045,
    right=0.955,
    top=0.82,
    bottom=0.13
)

# Add a small thematic heading.
fig.text(
    0.045, 0.965,
    "SWISS AGRICULTURAL STATISTICS",
    fontsize=12,
    fontweight="bold",
    color=PRIMARY,
    ha="left",
    va="top"
)

# Add the main infographic title.
fig.text(
    0.045, 0.925,
    "Laying and Breeding Chickens in Switzerland",
    fontsize=26,
    fontweight="bold",
    color=TEXT,
    ha="left",
    va="top"
)

# Add a descriptive subtitle.
fig.text(
    0.045, 0.880,
    f"National population trends ({start_year}-{end_year}) "
    f"and geographic distribution by canton ({reference_year})",
    fontsize=12,
    color=MUTED,
    ha="left",
    va="top"
)

# Include a subtle divider beneath the introductory text.
fig.add_artist(
    plt.Line2D(
        [0.045, 0.955],
        [0.845, 0.845],
        transform=fig.transFigure,
        color=BORDER,
        linewidth=1.2
    )
)


# -------------------------------------------------------------------
# 9.8. Incorporate the graphics and summary statistics
# -------------------------------------------------------------------

# Left panel: national time-series chart.
ax_line = fig.add_subplot(gs[0, 0])

ax_line.imshow(line_img)
ax_line.set_axis_off()

# Right panel: canton-level choropleth map.
ax_map = fig.add_subplot(gs[0, 1])

ax_map.imshow(map_img)
ax_map.set_axis_off()

# Create two summary panels beneath the graphics.
ax_national = fig.add_subplot(gs[1, 0])
ax_cantons = fig.add_subplot(gs[1, 1])

for ax in [ax_national, ax_cantons]:
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_axis_off()

    # Draw a subtle background around each summary panel.
    background = plt.Rectangle(
        (0, 0),
        1, 1,
        transform=ax.transAxes,
        facecolor=LIGHT,
        edgecolor=BORDER,
        linewidth=0.8,
        zorder=0
    )

    ax.add_patch(background)


# National summary panel.
ax_national.text(
    0.05, 0.86,
    "NATIONAL OVERVIEW",
    fontsize=12,
    fontweight="bold",
    color=PRIMARY,
    va="top"
)

ax_national.text(
    0.05, 0.67,
    f"{national_2025:,}",
    fontsize=27,
    fontweight="bold",
    color=BURGUNDY,
    va="top"
)

ax_national.text(
    0.05, 0.43,
    f"Laying and breeding chickens in {reference_year}",
    fontsize=11,
    color=MUTED,
    va="top"
)

ax_national.text(
    0.05, 0.20,
    f"Change from {start_year} to {end_year}: "
    f"{pct_change:+.1f}%",
    fontsize=12,
    fontweight="bold",
    color=TEXT,
    va="top"
)


# Canton summary panel.
ax_cantons.text(
    0.05, 0.86,
    "LEADING CANTONS IN 2025",
    fontsize=12,
    fontweight="bold",
    color=PRIMARY,
    va="top"
)

# Display the canton rankings on separate lines.
for i, row in top3.iterrows():

    canton_label = (
        f"{i + 1}. {row['canton_name']} "
        f"({row['canton']})"
    )

    canton_value = (
        f"{int(row[value_col]):,}"
    )

    y_position = 0.66 - (i * 0.19)

    ax_cantons.text(
        0.05, y_position,
        canton_label,
        fontsize=11,
        color=TEXT,
        va="center"
    )

    ax_cantons.text(
        0.95, y_position,
        canton_value,
        fontsize=11,
        fontweight="bold",
        color=BURGUNDY,
        ha="right",
        va="center"
    )

ax_cantons.text(
    0.05, 0.07,
    f"Combined share of national total: {top3_share:.1f}%",
    fontsize=11,
    fontweight="bold",
    color=BURGUNDY,
    va="bottom"
)


# -------------------------------------------------------------------
# 9.9. Add source information and methodological notes
# -------------------------------------------------------------------

# Include a source statement and explain how the canton data
# should be interpreted.

source_note = (
    "Source: Swiss agricultural statistical service. "
    "Author's calculations and visualization."
)

method_note = (
    "Note: Canton-level values represent absolute numbers of "
    "laying and breeding chickens, not population density or "
    "production intensity."
)

fig.text(
    0.045, 0.085,
    source_note,
    fontsize=10,
    color=MUTED,
    ha="left",
    va="center"
)

fig.text(
    0.045, 0.058,
    method_note,
    fontsize=9,
    color=MUTED,
    ha="left",
    va="center"
)


# -------------------------------------------------------------------
# 9.10. Export the final infographic
# -------------------------------------------------------------------

# Save a high-resolution PNG suitable for reports, presentations,
# and online publication.

fig.savefig(
    infographic_path,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.15,
    facecolor="white"
)

# Display the infographic in the Jupyter Notebook.
plt.show()

# Close the figure to avoid retaining unnecessary memory.
plt.close(fig)

print("\nPASS: Infographic generated successfully.")
print(f"Saved to: {infographic_path}")


### Interpretation and limitations

- **Scope.** The count concerns *laying and breeding chickens* as defined by the source dataset, not all poultry or egg output.
- **Time coverage.** Observations are not annually spaced throughout the entire 1975–2025 period; early years are snapshots.
- **Spatial scale.** Cantons are administrative units. Differences in geographic size, farm structure, and production specialization limit direct interpretation of raw counts as intensity.
- **Causality.** The charts document changes and geographic differences but do not identify their causes.
- **Reproducibility.** Numerical inputs are the two supplied CSV files. Geographic boundaries are independently sourced and validated using canton abbreviations. No missing values are imputed.

**Suggested next step for research:** If data on agricultural holdings or usable agricultural area are available by canton, calculate birds per holding or per hectare to complement this absolute-count map. Such measures address a different question and should not replace the counts shown here.

**Data citation note:** Add the original statistical-service dataset title, table identifier/URL, publication or retrieval date, and any counting-methodology documentation before final publication.